# functools.wraps を使って関数デコーレータを定義する

関数に適用できるデコレータの特別な構文があります。入力の引数や戻り値にアクセスして値を変更したり、例外を送出したりできます。これは、セマンティクス強化、デバッグ、関数登録に役立ちます。

例えば、関数呼び出しの引数と戻り値を出力したいとします。これは、再起関数で関数呼び出しの入れ子になったスタックをデバッグするときに、特に有用です。そういうデコーレータを *args と **kwargs と使って全パラメータをラップ↓関数に渡すことで次のように定義します。

In [2]:
def trace(func):
  def wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    print(f'{func.__name__}({args!r}, {kwargs!r}) ' f'-> {result!r}')
    return result
  return wrapper

このデコーレータを@記号を用いて関数に適用します。

In [3]:
# @ 記号はデコレータがラップする関数を引数として呼び出して、
# 戻り値を同じスコープのもともとの名前に代入することと等価
# fibonacci = trace(fibonacci) と同じ意味
@trace
def fibonacci(n):
  """Return the n-th Fibonacci number
  """
  if n in (0, 1):
    return n
  return (fibonacci(n - 2) + fibonacci(n - 1))

デコレータ付きの関数を呼び出すと、fibonacci を実行する前後でラッパーのコードが実行されます。
再帰スタックのレベルごとに引数と戻り値を出力します。

In [4]:
fibonacci(4)

fibonacci((0,), {}) -> 0
fibonacci((1,), {}) -> 1
fibonacci((2,), {}) -> 1
fibonacci((1,), {}) -> 1
fibonacci((0,), {}) -> 0
fibonacci((1,), {}) -> 1
fibonacci((2,), {}) -> 1
fibonacci((3,), {}) -> 2
fibonacci((4,), {}) -> 3


3

これはきちんと動作しますが、意図しない副作用があります。デコレータから返された値（上記で呼び出された関数）は、fibonacci という名前ではありません。

In [6]:
print(fibonacci)
print(fibonacci.__name__)

<function trace.<locals>.wrapper at 0x77ae9c22dda0>
wrapper


原因はそう難しくありません。trace 関数は、その本体で定義する wrapper を返します。デコレータの働きで、wrapper 関数が、定義元のモジュールでの fibonacci という名前に代入されたのです。この振る舞いは、デバッガのようなイントロスペクションを行うツールの機能を損なうので、問題があります。

例えば、デコレートされたfibonacci 関数には、組み込み関数 help が役立ちません。上で定義された docstring('Return the n-th Fibonacci number') を出力すべきです。

In [7]:
print(fibonacci.__doc__) # Return the n-th Fibonacci number が返ってきてほしい
help(fibonacci)

None
Help on function wrapper in module __main__:

wrapper(*args, **kwargs)



オブジェクトシリアライザーは、デコーレートされた元の関数の位置を決定できないのでエラーとなります。

In [9]:
import pickle

pickle.dumps(fibonacci)

AttributeError: Can't pickle local object 'trace.<locals>.wrapper'

解決法は、組み込みモジュール functions の wraps ヘルパー関数を使うことです。これは、デコレータを書くのを助けるデコーダーです。これを wrapper 関数に適用すると、内部関数についてのすべての重要なメタデータががうぶ関数に複製されます。

In [10]:
from functools import wraps

def trace_wrap(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    print(f'{func.__name__}({args!r}, {kwargs!r}) ' f'-> {result!r}')
    return result
  return wrapper

In [15]:
@trace_wrap
def fibonacci_a(n):
  """Return the n-th Fibonacci number
  """
  if n in (0, 1):
    return n
  return (fibonacci_a(n - 2) + fibonacci_a(n - 1))

help 関数を実行すると、関数がデコレートされているにも関わらず期待された結果が得られます。

In [16]:
print(fibonacci_a)
print(fibonacci_a.__name__)
print(fibonacci_a.__doc__)
help(fibonacci_a)

<function fibonacci_a at 0x77ae874de200>
fibonacci_a
Return the n-th Fibonacci number
  
Help on function fibonacci_a in module __main__:

fibonacci_a(n)
    Return the n-th Fibonacci number



In [17]:
# pickle オブジェクトシリアライザーも動作します。
print(pickle.dumps(fibonacci_a))

b'\x80\x04\x95\x1c\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x0bfibonacci_a\x94\x93\x94.'


これらの例の他にも、Python 関数は、言語の関数インタフェースを保守するために保持しなければならない多くの標準属性（例えば、\_\_name\_\_, \_\_module\_\_, \_\_annotations\_\_）を持っています。wraps を使うことで、正しい振る舞いが得られます。

## 覚えておくこと

- Python のデコレータ構文を使うと、ある関数が他の関数を実行時に修正できる。
- デコレータを使うことでデバッガのようなイントロスペクションを行うツールに奇妙な振る舞いを引き起こすことがある。
- 問題を引き起こさないようにデコレータを自分で定義するときには、組み込みモジュール functools のデコレータ wraps を使う。